# Patcher LoRA fine-tune (Google Colab · A100)

Trains a **unified-diff patcher** on JSONL produced by `build_patcher_data.py` (`task_type=patch_generation`, `input` JSON + `output.unified_diff`).

**Before you run:**
- **Runtime → Disconnect and delete runtime** before every fresh run to avoid cached bf16 model weights leaking into a QLoRA session.
- **Runtime → Change runtime type → GPU → A100** (paid tier / availability varies).
- Upload `patcher_train.jsonl` and `patcher_eval.jsonl` under Drive:  
  `/content/drive/MyDrive/autobot_patcher/outputs/` (same layout as local `training/patch_patcher/outputs/`).
- Add a Colab secret **`HF_TOKEN`** (Hugging Face read token) so the base model `Qwen/Qwen2.5-Coder-7B-Instruct` can download.

**Key design decisions:**
- `MAX_LENGTH=6144` — your dataset p50 is ~4,279 tokens; 6144 covers 82.7% of rows completely. Lower values (2048/3072) silently truncate the output diff on 85–98% of rows, giving the model almost no learning signal.
- `LOAD_IN_4BIT=True` — mandatory at 6144 token sequences; drops base model from 14 GB to ~3.5 GB.
- `liger-kernel` — replaces `logits.float()` upcast in the loss with a chunked fused CE kernel. This is the root cause of every previous OOM; no trainer flag can disable it without this patch.
- `DISABLE_EVAL_DURING_TRAINING=True` — VRAM fragmentation after ~50 training steps makes the eval forward pass OOM even when training is stable. Run eval manually post-training on a fresh GPU state.

## Cell 1 — Install dependencies

- **`torchao>=0.16.0`**: Colab often ships an older `torchao`; PEFT's LoRA path expects a recent version.
- **`liger-kernel`**: Patches Qwen2's CE loss to avoid the `logits.float()` OOM spike.
- After this cell completes, use **Runtime → Restart session**, then run from **Cell 2** onward.

In [1]:
# ============================================================
# CELL 1 — pip installs (run once per fresh runtime)
# ============================================================
!pip install -q --upgrade "torchao>=0.16.0"
!pip install -q transformers peft accelerate bitsandbytes datasets trl huggingface_hub
!pip install -q liger-kernel

# torch is preinstalled on Colab; keep installs minimal to avoid ABI mismatches.
# After this cell: Runtime → Restart session, then run from Cell 2 onward.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 119.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.1/403.1 kB 34.5 MB/s eta 0:00:00


## Restart runtime

**Runtime → Restart session** so the upgraded `torchao` and `liger-kernel` are actually imported.

In [1]:
# ============================================================
# CELL 2 — Set allocator env var BEFORE any CUDA init
# ============================================================
# MUST come before 'import torch' — once CUDA is initialised this flag is ignored.
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("PYTORCH_CUDA_ALLOC_CONF set:", os.environ["PYTORCH_CUDA_ALLOC_CONF"])

PYTORCH_CUDA_ALLOC_CONF set: expandable_segments:True


In [2]:
# ============================================================
# CELL 3 — Mount Google Drive
# ============================================================
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
# ============================================================
# CELL 4 — Imports, paths, and dataset helpers
# ============================================================
# Same data layout as local build_patcher_data.py OUTPUT_DIR:
#   .../outputs/patcher_train.jsonl
#   .../outputs/patcher_eval.jsonl

from __future__ import annotations

import json
import logging
import math
import os
import random
import sys
from pathlib import Path

import torch
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

LOG = logging.getLogger("patcher_colab")
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

# --- Paths (edit if your Drive layout differs) ---
DRIVE_PATCHER = Path("/content/drive/MyDrive/autobot_patcher")
JSONL_DIR = DRIVE_PATCHER / "outputs"
TRAIN_JSONL = JSONL_DIR / "patcher_train.jsonl"
EVAL_JSONL = JSONL_DIR / "patcher_eval.jsonl"
OUTPUT_DIR = DRIVE_PATCHER / "patcher_sft_run_a100"
ADAPTER_DIR = DRIVE_PATCHER / "patcher_lora_adapter_a100"

BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"


def _require_torchao_compatible_for_peft() -> None:
    """Raise if torchao is too old for the installed peft (common on Colab before upgrade + restart)."""
    try:
        import importlib.metadata as im
    except ImportError:
        return
    try:
        raw = im.version("torchao")
    except im.PackageNotFoundError:
        return
    main_ver = raw.split("+", 1)[0].strip()
    nums: list[int] = []
    for part in main_ver.split("."):
        try:
            nums.append(int(part))
        except ValueError:
            return
    while len(nums) < 3:
        nums.append(0)
    if tuple(nums[:3]) >= (0, 16, 0):
        return
    raise RuntimeError(
        f"torchao {raw!r} is too old for peft (need >=0.16.0). "
        'pip install "torchao>=0.16.0" then Runtime → Restart session.'
    )


_PATCHER_SYSTEM = (
    "You are AutoBot Patcher. Respond with a unified diff patch only.\n\n"
    "Scope and context come from the user's JSON payload:\n"
    "- instruction: simulator instruction (follow it).\n"
    "- planner_directive, issue_context, file_contexts, allowed_edit_files, "
    "treesitter_context, graphrag_context, constraints: structured context aligned "
    "with patcher inference.\n\n"
    "Rules:\n"
    "- Output ONLY unified diff text (GNU diff unified format). "
    "No markdown fences, no commentary, no field labels.\n"
    "- Touch only paths that appear under allowed_edit_files / constraints.\n"
    "- Keep the change minimal and consistent with planner scope."
)


def _read_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open(encoding="utf-8") as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError as e:
                raise ValueError(f"{path}:{i+1} invalid JSON: {e}") from e
            rows.append(obj)
    return rows


def _prompt_for_row(row: dict, tokenizer: AutoTokenizer) -> tuple[str, str]:
    inp = row.get("input") or {}
    out = row.get("output") or {}
    diff = out.get("unified_diff") or ""
    if not isinstance(diff, str):
        raise TypeError(f"row {row.get('id')}: output.unified_diff must be str")
    if not inp:
        raise ValueError(f"row {row.get('id')}: missing input")

    user_text = json.dumps(inp, ensure_ascii=False)
    messages = [
        {"role": "system", "content": _PATCHER_SYSTEM},
        {"role": "user", "content": user_text},
    ]

    templ = getattr(tokenizer, "chat_template", None)
    if templ:
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        _im_end = "<|im" + "_end|>"
        qwen_turn_end = tokenizer.eos_token or _im_end
        prompt = (
            "<|im_start|>system\n"
            + _PATCHER_SYSTEM
            + "\n"
            + qwen_turn_end
            + "\n"
            "<|im_start|>user\n"
            + user_text
            + "\n"
            + qwen_turn_end
            + "\n"
            "<|im_start|>assistant\n"
        )

    end_tok = tokenizer.eos_token or ""
    completion = diff.rstrip("\n") + (end_tok if end_tok else "")
    return prompt, completion


def _records_to_hf_dataset(rows: list[dict], tokenizer: AutoTokenizer) -> Dataset:
    prompts, completions = [], []
    for row in rows:
        if row.get("task_type") not in {None, "patch_generation"}:
            continue
        p, c = _prompt_for_row(row, tokenizer)
        prompts.append(p)
        completions.append(c)
    return Dataset.from_dict({"prompt": prompts, "completion": completions})


def _estimate_warmup_steps(
    n_train: int,
    per_device_bs: int,
    grad_accum: int,
    num_epochs: float,
    warmup_ratio: float,
) -> int:
    gpus = max(1, int(os.environ.get("LOCAL_WORLD_SIZE", os.environ.get("WORLD_SIZE", "1"))))
    eff = max(1, per_device_bs * grad_accum * gpus)
    steps_per_epoch = max(1, math.ceil(n_train / eff))
    total = max(1, int(steps_per_epoch * num_epochs))
    return max(1, int(total * warmup_ratio))


_require_torchao_compatible_for_peft()
print("Helpers loaded; JSONL dir:", JSONL_DIR)

Helpers loaded; JSONL dir: /content/drive/MyDrive/autobot_patcher/outputs


In [4]:
# ============================================================
# CELL 5 — Hugging Face login
# ============================================================
# Colab: Project settings (left sidebar) → Secrets → add HF_TOKEN (HF read access).

from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

In [5]:
# ============================================================
# CELL 6 — Training hyperparameters
# ============================================================
#
# MAX_LENGTH=6144: your dataset p50 is ~4,279 tokens. Coverage by MAX_LENGTH:
#   2048 ->  1.6%  of rows fit (97.4% of diffs fully truncated — no learning signal)
#   3072 -> 14.5%  of rows fit
#   4096 -> 45.5%  of rows fit
#   6144 -> 82.7%  of rows fit  <- current setting
#   8192 -> 96.8%  of rows fit  <- ideal but slower
#
# LOAD_IN_4BIT=True: mandatory at 6144 tokens. Drops base model 14 GB -> ~3.5 GB.
# DISABLE_EVAL_DURING_TRAINING=True: VRAM fragmentation after ~50 steps causes eval
#   to OOM even when training is stable. Run eval manually after training completes.

SEED = 42
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8     # effective batch 8; reduced from 16 for 6144-token headroom
LEARNING_RATE = 5e-5
WARMUP_RATIO = 0.1
NUM_TRAIN_EPOCHS = 3.0
LOGGING_STEPS = 10
EVAL_STEPS = 50
SAVE_STEPS = 50
SAVE_TOTAL_LIMIT = 2
MAX_LENGTH = 6144                    # covers 82.7% of dataset rows completely
DATALOADER_NUM_WORKERS = 2           # reduced from 4; long seqs need more CPU RAM per worker
GRADIENT_CHECKPOINTING = True

# LoRA
LORA_R = 32
LORA_ALPHA = 128
LORA_DROPOUT = 0.05
LORA_ONLY = True                     # no lm_head/embed_tokens in modules_to_save; saves VRAM

# Memory
LOAD_IN_4BIT = True                  # mandatory — do Runtime → Disconnect and delete runtime first
ADAM_8BIT = True                     # halves optimizer state VRAM

# Subsampling
TRAIN_SUBSAMPLE_MAX = 2000           # set None to train on all 18,885 rows
EVAL_SUBSAMPLE_MAX = 256             # caps rows used during in-training eval; JSONL on disk unchanged

# Disable in-training eval to avoid VRAM fragmentation OOM at step 50.
# Set False only if liger-kernel is confirmed working and you want live eval_loss.
DISABLE_EVAL_DURING_TRAINING = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

assert TRAIN_JSONL.is_file(), f"Missing {TRAIN_JSONL} — copy JSONL from build_patcher_data outputs"
eval_exists = EVAL_JSONL.is_file()
if not eval_exists:
    LOG.warning("No eval file at %s — training without validation.", EVAL_JSONL)

print("OUTPUT_DIR:", OUTPUT_DIR)
print("ADAPTER_DIR:", ADAPTER_DIR)
print("train:", TRAIN_JSONL, "eval:", EVAL_JSONL if eval_exists else "(none)")
print(f"MAX_LENGTH={MAX_LENGTH}  LOAD_IN_4BIT={LOAD_IN_4BIT}  DISABLE_EVAL={DISABLE_EVAL_DURING_TRAINING}")

OUTPUT_DIR: /content/drive/MyDrive/autobot_patcher/patcher_sft_run_a100
ADAPTER_DIR: /content/drive/MyDrive/autobot_patcher/patcher_lora_adapter_a100
train: /content/drive/MyDrive/autobot_patcher/outputs/patcher_train.jsonl eval: /content/drive/MyDrive/autobot_patcher/outputs/patcher_eval.jsonl
MAX_LENGTH=6144  LOAD_IN_4BIT=True  DISABLE_EVAL=True


In [6]:
# ============================================================
# CELL 7 — GPU checks + TF32 (Ampere A100)
# ============================================================
# Note: PYTORCH_CUDA_ALLOC_CONF was set in Cell 2 before CUDA init.

if not torch.cuda.is_available():
    raise RuntimeError("CUDA not available — select GPU runtime (prefer A100).")

print(torch.cuda.get_device_name(0))
print("Alloc conf:", os.environ.get("PYTORCH_CUDA_ALLOC_CONF", "NOT SET — re-run Cell 2"))

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
LOG.info("TF32 enabled for matmul (A100-friendly).")

NVIDIA A100-SXM4-80GB
Alloc conf: expandable_segments:True


In [7]:
# ============================================================
# CELL 8 — Tokenizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
# ============================================================
# CELL 9 — Load JSONL → HF Datasets
# ============================================================

raw_train = _read_jsonl(TRAIN_JSONL)
eval_rows = _read_jsonl(EVAL_JSONL) if eval_exists else []
if not raw_train:
    raise RuntimeError(f"No rows in {TRAIN_JSONL}")

patch_train = [r for r in raw_train if r.get("task_type") in {None, "patch_generation"}]
if TRAIN_SUBSAMPLE_MAX is not None and len(patch_train) > TRAIN_SUBSAMPLE_MAX:
    rng = random.Random(SEED)
    rng.shuffle(patch_train)
    train_rows = patch_train[:TRAIN_SUBSAMPLE_MAX]
    LOG.info(
        "Train subsample: %d patch_generation rows (cap TRAIN_SUBSAMPLE_MAX=%s; had %d)",
        len(train_rows),
        TRAIN_SUBSAMPLE_MAX,
        len(patch_train),
    )
else:
    train_rows = patch_train

train_ds = _records_to_hf_dataset(train_rows, tokenizer)
eval_ds = _records_to_hf_dataset(eval_rows, tokenizer)

if len(train_ds) == 0:
    raise RuntimeError("No patch_generation rows after filtering")

if (
    len(eval_ds) > 0
    and EVAL_SUBSAMPLE_MAX is not None
    and len(eval_ds) > EVAL_SUBSAMPLE_MAX
):
    eval_ds = eval_ds.shuffle(seed=SEED).select(range(EVAL_SUBSAMPLE_MAX))
    LOG.info("Eval subsample for training: %d rows (EVAL_SUBSAMPLE_MAX=%s)", len(eval_ds), EVAL_SUBSAMPLE_MAX)

print(f"Train samples: {len(train_ds)} | Eval samples: {len(eval_ds)}")

Train samples: 2000 | Eval samples: 256


In [12]:
# ============================================================
# CELL 10 — Load base model + patch loss to avoid logits.float() OOM
# ============================================================
# Liger Kernel is incompatible with completion_only_loss=True (it returns
# logits=None; TRL then crashes accessing outputs.logits for prompt masking).
# Instead we monkey-patch ForCausalLMLoss to skip the float32 upcast and
# compute cross-entropy directly in bf16. Same memory saving, no conflict.

import torch.nn.functional as F
import transformers.loss.loss_utils as _lu

def _bf16_ce_loss(logits, labels, vocab_size, num_items_in_batch=None,
                  ignore_index=-100, shift_labels=None, **kwargs):
    if shift_labels is None:
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels_t = labels[..., 1:].contiguous()
    else:
        shift_logits = logits
        shift_labels_t = shift_labels
    shift_logits = shift_logits.view(-1, vocab_size)
    shift_labels_t = shift_labels_t.view(-1).to(shift_logits.device)
    reduction = "sum" if num_items_in_batch is not None else "mean"
    loss = F.cross_entropy(
        shift_logits, shift_labels_t,
        ignore_index=ignore_index, reduction=reduction
    )
    if num_items_in_batch is not None:
        loss = loss / num_items_in_batch
    return loss

_lu.ForCausalLMLoss = _bf16_ce_loss
LOG.info("ForCausalLMLoss patched: bf16 CE, no logits.float() upcast.")

dtype = torch.bfloat16

if LOAD_IN_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model = prepare_model_for_kbit_training(
        model, use_gradient_checkpointing=GRADIENT_CHECKPOINTING
    )
else:
    try:
        model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL, dtype=dtype, device_map="auto", trust_remote_code=True
        )
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL, torch_dtype=dtype, device_map="auto", trust_remote_code=True
        )

model.config.use_cache = False
model.resize_token_embeddings(len(tokenizer))

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Embedding(151665, 3584)

In [13]:
# ============================================================
# CELL 11 — Attach LoRA
# ============================================================

_lora_common = dict(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj"],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
if not LORA_ONLY:
    _lora_common["modules_to_save"] = ["lm_head", "embed_tokens"]

try:
    lora_cfg = LoraConfig(**_lora_common, ensure_weight_tying=True)
except TypeError:
    lora_cfg = LoraConfig(**_lora_common)

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

if GRADIENT_CHECKPOINTING and not LOAD_IN_4BIT:
    model.enable_input_require_grads()
    model.gradient_checkpointing_enable()
elif GRADIENT_CHECKPOINTING and LOAD_IN_4BIT:
    model.enable_input_require_grads()

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1332: UserWarning: You have requested `ensure_weight_tying`, but no tied modules are added in either `modules_to_save` or `target_modules`
  warnings.warn(


trainable params: 60,555,264 || all params: 7,673,311,744 || trainable%: 0.7892


In [14]:
# ============================================================
# CELL 12 — SFTConfig + SFTTrainer + train + save
# ============================================================
# completion_only_loss=True masks the prompt; loss is only on the unified diff tokens.
# DISABLE_EVAL_DURING_TRAINING=True skips in-training eval checkpoints entirely;
# run eval manually post-training using the saved adapter on a fresh GPU state.

import gc

warmup_steps = _estimate_warmup_steps(
    len(train_ds),
    PER_DEVICE_TRAIN_BATCH_SIZE,
    GRADIENT_ACCUMULATION_STEPS,
    NUM_TRAIN_EPOCHS,
    WARMUP_RATIO,
)
LOG.info("Warmup steps (estimated): %s", warmup_steps)

# use_eval: only enable eval path if we have data AND eval is not disabled
use_eval = len(eval_ds) > 0 and not DISABLE_EVAL_DURING_TRAINING

cfg_kwargs = dict(
    output_dir=str(OUTPUT_DIR),
    seed=SEED,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,
    logging_steps=LOGGING_STEPS,
    max_length=MAX_LENGTH,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    greater_is_better=False,
    report_to="none",
    completion_only_loss=True,
    packing=False,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
    dataloader_num_workers=DATALOADER_NUM_WORKERS,
    tf32=True,
    bf16=True,
)

if use_eval:
    cfg_kwargs["bf16_full_eval"] = True
if ADAM_8BIT:
    cfg_kwargs["optim"] = "adamw_bnb_8bit"
if use_eval:
    cfg_kwargs.update(
        eval_strategy="steps",
        eval_steps=EVAL_STEPS,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
    )
else:
    cfg_kwargs.update(eval_strategy="no", load_best_model_at_end=False)

try:
    training_args = SFTConfig(**cfg_kwargs)
except TypeError:
    cfg_kwargs.pop("warmup_steps", None)
    cfg_kwargs["warmup_ratio"] = WARMUP_RATIO
    try:
        training_args = SFTConfig(**cfg_kwargs)
    except TypeError:
        cfg_kwargs.pop("bf16_full_eval", None)
        training_args = SFTConfig(**cfg_kwargs)

common = dict(model=model, args=training_args, train_dataset=train_ds)
try:
    trainer = SFTTrainer(
        **common,
        eval_dataset=eval_ds if use_eval else None,
        processing_class=tokenizer,
    )
except TypeError:
    trainer = SFTTrainer(
        **common,
        eval_dataset=eval_ds if use_eval else None,
        tokenizer=tokenizer,
    )

LOG.info("Starting training... eval_during_training=%s", use_eval)
gc.collect()
torch.cuda.empty_cache()
trainer.train()

trainer.model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
LOG.info("Saved adapter + tokenizer to %s", ADAPTER_DIR)

Adding EOS to train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,0.201006
20,0.142545
30,0.122518
40,0.156449
50,0.109941
60,0.113406
70,0.080615
80,0.078062
90,0.088089
100,0.150756


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetunin

## Cell 13 — Manual eval on saved adapter (run after training)

Because `DISABLE_EVAL_DURING_TRAINING=True`, run this cell after training to get eval loss on the full eval set. It loads the saved adapter fresh, so VRAM is clean and fragmentation is not an issue.

In [15]:
!pip install -q openai

In [21]:
# ============================================================
# CELL 13 — PATCHER EVAL: FULL PERFORMANCE AUDIT
# ============================================================
# Run this in a FRESH runtime after training completes.
# Steps:
#   1. Runtime → Disconnect and delete runtime
#   2. Runtime → Change runtime type → GPU → A100
#   3. Re-run: Cell 2 (allocator), Cell 3 (drive mount),
#              Cell 4 (imports/helpers), Cell 5 (HF login),
#              Cell 8 (tokenizer), then this cell.
#
# Requires: OPENAI_API_KEY in Colab Secrets (same place as HF_TOKEN)
#           !pip install -q openai  (add to Cell 1 if not present)
#
# Metrics (Base vs Fine-Tuned):
#   - Valid diff format       : output is parseable unified diff
#   - No prose                : no markdown fences / commentary lines
#   - File recall             : predicted touched files ∩ gold / gold
#   - Scope compliance        : predicted files ⊆ allowed_edit_files
#   - Hunk header overlap     : predicted diff contains gold @@ headers
#   - G-Eval (OpenAI, 1-5):
#       instruction_adherence : implements the right change
#       scope_compliance      : no unrelated file edits
#       diff_quality          : valid, minimal, semantically close to gold
#       syntax_and_style      : valid Python, correct indentation
#       hallucination_penalty : no invented symbols / imports / APIs
#   - Generation latency      : wall-clock per generate() call
#
# Outputs saved to Drive:
#   patcher_eval_detailed_report.json
#   patcher_eval_metrics_comparison.json
#   patcher_eval_run_meta.json
#   patcher_metrics_readout.txt
#   patcher_per_row_latency.csv
# ============================================================

import gc
import json
import os
import re
import datetime
import shutil
import statistics
import time
from pathlib import Path

import pandas as pd
import torch
import torch.nn.functional as F
import transformers.loss.loss_utils as _lu
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from tqdm.auto import tqdm
from openai import OpenAI
from google.colab import userdata as _userdata

# ── config ───────────────────────────────────────────────────
DRIVE_PATCHER = Path("/content/drive/MyDrive/autobot_patcher")
EVAL_JSONL    = DRIVE_PATCHER / "outputs" / "patcher_eval.jsonl"
ADAPTER_DIR   = DRIVE_PATCHER / "patcher_lora_adapter_a100"
BASE_MODEL    = "Qwen/Qwen2.5-Coder-7B-Instruct"
EVAL_OUT_DIR  = DRIVE_PATCHER / "evals" / "patcher" / "v1"
_GEVAL_MODEL  = "gpt-4o-mini"

# 150 rows ≈ 30-40 min on A100 for both Base + FT. None = all 2360.
MAX_EVAL_ROWS = 150

# ── reset output dir ─────────────────────────────────────────
if EVAL_OUT_DIR.is_dir():
    shutil.rmtree(EVAL_OUT_DIR)
EVAL_OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Eval output directory (reset): {EVAL_OUT_DIR}")

# ── load eval JSONL ──────────────────────────────────────────
eval_rows = []
with EVAL_JSONL.open(encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            eval_rows.append(json.loads(line))

if MAX_EVAL_ROWS is not None:
    eval_rows = eval_rows[:MAX_EVAL_ROWS]

n_total = len(eval_rows)
print(f"Eval rows: {n_total}  (full eval JSONL has 2360; cap MAX_EVAL_ROWS={MAX_EVAL_ROWS})")

# ── bf16 CE patch (same as training) ─────────────────────────
def _bf16_ce_loss(logits, labels, vocab_size, num_items_in_batch=None,
                  ignore_index=-100, shift_labels=None, **kwargs):
    if shift_labels is None:
        shift_logits  = logits[..., :-1, :].contiguous()
        shift_labels_t = labels[..., 1:].contiguous()
    else:
        shift_logits  = logits
        shift_labels_t = shift_labels
    shift_logits   = shift_logits.view(-1, vocab_size)
    shift_labels_t = shift_labels_t.view(-1).to(shift_logits.device)
    reduction = "sum" if num_items_in_batch is not None else "mean"
    loss = F.cross_entropy(shift_logits, shift_labels_t,
                           ignore_index=ignore_index, reduction=reduction)
    if num_items_in_batch is not None:
        loss = loss / num_items_in_batch
    return loss

_lu.ForCausalLMLoss = _bf16_ce_loss

# ── load model + adapter ─────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb_config,
    device_map="auto", trust_remote_code=True,
)
base_model.config.use_cache = False

TOKENIZER = AutoTokenizer.from_pretrained(str(ADAPTER_DIR), trust_remote_code=True)
TOKENIZER.pad_token = TOKENIZER.pad_token or TOKENIZER.eos_token
TOKENIZER.padding_side = "right"

# resize must match training (adapter was saved with resize active)
base_model.resize_token_embeddings(len(TOKENIZER))

ft_model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))
ft_model.eval()
print("Loaded FT adapter from", ADAPTER_DIR)

# ── prompt builder (mirrors training Cell 4) ─────────────────
_PATCHER_SYSTEM = (
    "You are AutoBot Patcher. Respond with a unified diff patch only.\n\n"
    "Scope and context come from the user's JSON payload:\n"
    "- instruction: simulator instruction (follow it).\n"
    "- planner_directive, issue_context, file_contexts, allowed_edit_files, "
    "treesitter_context, graphrag_context, constraints: structured context aligned "
    "with patcher inference.\n\n"
    "Rules:\n"
    "- Output ONLY unified diff text (GNU diff unified format). "
    "No markdown fences, no commentary, no field labels.\n"
    "- Touch only paths that appear under allowed_edit_files / constraints.\n"
    "- Keep the change minimal and consistent with planner scope."
)

def _build_prompt(row: dict) -> str:
    user_text = json.dumps(row["input"], ensure_ascii=False)
    messages  = [
        {"role": "system", "content": _PATCHER_SYSTEM},
        {"role": "user",   "content": user_text},
    ]
    templ = getattr(TOKENIZER, "chat_template", None)
    if templ:
        return TOKENIZER.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    _im_end = "<|im_end|>"
    return (
        f"<|im_start|>system\n{_PATCHER_SYSTEM}\n{_im_end}\n"
        f"<|im_start|>user\n{user_text}\n{_im_end}\n"
        "<|im_start|>assistant\n"
    )

# ── metric helpers ────────────────────────────────────────────
def extract_touched_files(diff_text: str) -> set:
    return set(re.findall(r"^\+\+\+ b/(.+)$", diff_text, re.MULTILINE))

def is_valid_diff(diff_text: str) -> bool:
    t = diff_text.strip()
    if not t:
        return False
    return bool(re.search(r"^(--- a/|diff --git )", t, re.MULTILINE))

def has_prose(diff_text: str) -> bool:
    if "```" in diff_text:
        return True
    non_diff = [
        l for l in diff_text.splitlines()
        if l.strip() and not l.startswith(
            ("-", "+", " ", "@", "\\", "d", "i", "n", "o")
        )
    ]
    return len(non_diff) > 3

def file_recall(pred_files: set, gold_files: list):
    if not gold_files:
        return None
    return len(pred_files & set(gold_files)) / len(gold_files)

def scope_compliance(pred_files: set, allowed_files: list) -> float:
    if not pred_files:
        return 1.0
    allowed = set(allowed_files)
    return len(pred_files & allowed) / len(pred_files)

def hunk_header_overlap(pred_diff: str, gold_hunks: list):
    if not gold_hunks:
        return None
    found = sum(1 for h in gold_hunks if h in pred_diff)
    return found / len(gold_hunks)

# ── generation ────────────────────────────────────────────────
def generate_prediction(prompt: str, use_adapter: bool,
                        max_new_tokens: int = 512) -> str:
    inputs = TOKENIZER(
        prompt, return_tensors="pt",
        truncation=True, max_length=6144,
    ).to(ft_model.device)
    with torch.no_grad():
        if use_adapter:
            out = ft_model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                do_sample=False, temperature=1.0,
            )
        else:
            with ft_model.disable_adapter():
                out = ft_model.generate(
                    **inputs, max_new_tokens=max_new_tokens,
                    do_sample=False, temperature=1.0,
                )
    return TOKENIZER.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )

# ── G-Eval (5 rubrics) ───────────────────────────────────────
def _openai_client() -> OpenAI:
    key = _userdata.get("OPENAI_API_KEY")
    if not key or not key.strip():
        raise RuntimeError("OPENAI_API_KEY secret missing in Colab Secrets.")
    return OpenAI(api_key=key.strip())

def geval_patcher(client, instruction: str, gold_diff: str,
                  pred_diff: str, file_context: str = "") -> dict:
    """
    5-rubric G-Eval for unified diff quality.

    instruction_adherence (1-5):
        Does the predicted diff implement the same change described
        in the instruction as the reference?

    scope_compliance (1-5):
        Does the predicted diff touch only files relevant to the
        instruction, with no unrelated edits?

    diff_quality (1-5):
        Is the predicted diff syntactically valid unified diff, minimal,
        and semantically close to the reference change?

    syntax_and_style (1-5):
        Are the predicted additions valid Python code? Does the
        indentation perfectly match surrounding context lines?
        Penalize heavily for missing colons, unbalanced brackets,
        wrong indentation depth, or whitespace errors that would cause
        git apply to succeed but the code to crash immediately.

    hallucination_penalty (1-5):
        Did the model invent variables, functions, import paths, or
        external API calls NOT present in the reference diff or the
        provided file context?
        5 = strictly factual, no inventions.
        1 = heavy hallucinations (made-up symbols, non-existent modules).
    """
    system = (
        "You are a strict evaluator for an automated code patcher (G-Eval style). "
        "Score the MODEL OUTPUT unified diff against the REFERENCE diff. "
        "Each score is an integer 1 (poor) to 5 (excellent). "
        "Reply ONLY with a JSON object, no prose."
    )
    user = f"""Score the MODEL OUTPUT diff against the REFERENCE diff.

Dimensions (each 1-5):
- instruction_adherence: Does the predicted diff implement the same change described in the instruction as the reference?
- scope_compliance: Does the predicted diff touch only files relevant to the instruction, with no unrelated edits?
- diff_quality: Is the predicted diff syntactically valid unified diff, minimal, and semantically close to the reference change?
- syntax_and_style: Are the predicted additions valid Python code? Does the indentation perfectly match the surrounding context lines? Penalize heavily for missing colons, unbalanced brackets, wrong indentation depth, or trailing whitespace errors that would cause git apply to succeed but the code to crash immediately.
- hallucination_penalty: Did the model invent variables, functions, import paths, or external API calls that are NOT present in the reference diff or the provided file context? Score 5 = strictly factual, no inventions. Score 1 = heavy hallucinations (made-up symbols, non-existent modules, fabricated method names).

INSTRUCTION:
{instruction[:600]}

FILE CONTEXT (for hallucination check):
{file_context[:800]}

REFERENCE DIFF (gold):
{gold_diff[:1200]}

MODEL OUTPUT DIFF:
{pred_diff[:1200]}

Return only JSON: {{"instruction_adherence": <int>, "scope_compliance": <int>, "diff_quality": <int>, "syntax_and_style": <int>, "hallucination_penalty": <int>}}"""

    resp = client.chat.completions.create(
        model=_GEVAL_MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ],
        temperature=0,
        max_tokens=150,
    )
    raw = (resp.choices[0].message.content or "").strip()
    if raw.startswith("```"):
        raw = re.sub(r"^```(?:json)?\s*", "", raw, flags=re.IGNORECASE)
        raw = re.sub(r"\s*```$", "", raw).strip()
    d  = json.loads(raw)
    ia = max(1, min(5, int(d.get("instruction_adherence", 3))))
    sc = max(1, min(5, int(d.get("scope_compliance",      3))))
    dq = max(1, min(5, int(d.get("diff_quality",          3))))
    ss = max(1, min(5, int(d.get("syntax_and_style",      3))))
    hp = max(1, min(5, int(d.get("hallucination_penalty", 3))))
    return {
        "instruction_adherence": ia,
        "scope_compliance":      sc,
        "diff_quality":          dq,
        "syntax_and_style":      ss,
        "hallucination_penalty": hp,
        "mean": round((ia + sc + dq + ss + hp) / 5.0, 4),
    }

# ── main eval loop ────────────────────────────────────────────
metrics = {
    mode: {
        "valid_diff": [], "no_prose": [], "file_recall": [],
        "scope_compliance": [], "hunk_overlap": [], "gen_sec": [],
        "geval": [], "texts": [],
    }
    for mode in ["FT", "Base"]
}

gc.collect()
torch.cuda.empty_cache()

print(f"\nRunning Base vs FT on {n_total} eval rows...\n")
for row in tqdm(eval_rows, total=n_total):
    prompt      = _build_prompt(row)
    gold_diff   = row["output"]["unified_diff"]
    gold_files  = row["output"]["touched_files"]
    gold_hunks  = row["output"]["gold_hunk_headers"]
    allowed     = row["input"]["allowed_edit_files"]

    for mode in ["FT", "Base"]:
        t0   = time.perf_counter()
        pred = generate_prediction(prompt, use_adapter=(mode == "FT"))
        metrics[mode]["gen_sec"].append(time.perf_counter() - t0)
        metrics[mode]["texts"].append(pred)

        pred_files = extract_touched_files(pred)
        metrics[mode]["valid_diff"].append(int(is_valid_diff(pred)))
        metrics[mode]["no_prose"].append(int(not has_prose(pred)))
        metrics[mode]["file_recall"].append(file_recall(pred_files, gold_files))
        metrics[mode]["scope_compliance"].append(scope_compliance(pred_files, allowed))
        metrics[mode]["hunk_overlap"].append(hunk_header_overlap(pred, gold_hunks))

# ── G-Eval pass ───────────────────────────────────────────────
print("\nRunning G-Eval (OpenAI, 5 rubrics) on predictions...")
_client = _openai_client()

for mode in ["FT", "Base"]:
    for i, row in enumerate(tqdm(eval_rows, desc=f"G-Eval {mode}")):
        pred         = metrics[mode]["texts"][i]
        instruction  = row["input"].get("instruction", "")
        gold_diff    = row["output"]["unified_diff"]
        file_context = json.dumps(
            row["input"].get("file_contexts", {}).get("primary", {}),
            ensure_ascii=False,
        )[:800]
        try:
            scores = geval_patcher(
                _client, instruction, gold_diff, pred, file_context
            )
        except Exception as e:
            print(f"G-Eval row {i} failed ({mode}): {e}")
            scores = {
                "instruction_adherence": float("nan"),
                "scope_compliance":      float("nan"),
                "diff_quality":          float("nan"),
                "syntax_and_style":      float("nan"),
                "hallucination_penalty": float("nan"),
                "mean":                  float("nan"),
            }
        metrics[mode]["geval"].append(scores)
        time.sleep(0.05)

# ── summary table ─────────────────────────────────────────────
def _mean(lst):
    vals = [v for v in lst if v is not None and v == v]
    return sum(vals) / len(vals) if vals else 0.0

def _geval_dim_mean(mode, dim):
    return _mean([g[dim] for g in metrics[mode]["geval"]])

summary_results = []
for mode in ["Base", "FT"]:
    m = metrics[mode]
    summary_results.append({
        "Model":                   "Fine-Tuned (AutoBot Patcher)" if mode == "FT" else "Base (Qwen 2.5 Coder)",
        "Valid Diff %":            f"{_mean(m['valid_diff']):.2%}",
        "No Prose %":              f"{_mean(m['no_prose']):.2%}",
        "File Recall":             f"{_mean([v for v in m['file_recall'] if v is not None]):.4f}",
        "Scope Compliance":        f"{_mean(m['scope_compliance']):.4f}",
        "Hunk Overlap":            f"{_mean([v for v in m['hunk_overlap'] if v is not None]):.4f}",
        "G-Eval mean (1-5)":       f"{_mean([g['mean'] for g in m['geval'] if g['mean']==g['mean']]):.4f}",
        "  instruction_adherence": f"{_geval_dim_mean(mode, 'instruction_adherence'):.4f}",
        "  scope_compliance":      f"{_geval_dim_mean(mode, 'scope_compliance'):.4f}",
        "  diff_quality":          f"{_geval_dim_mean(mode, 'diff_quality'):.4f}",
        "  syntax_and_style":      f"{_geval_dim_mean(mode, 'syntax_and_style'):.4f}",
        "  hallucination_penalty": f"{_geval_dim_mean(mode, 'hallucination_penalty'):.4f}",
        "Gen mean (s)":            f"{_mean(m['gen_sec']):.2f}",
    })

print("\n" + "=" * 80)
print("PATCHER EVAL — FINAL PERFORMANCE COMPARISON")
print("=" * 80)
print(pd.DataFrame(summary_results).to_string(index=False))

# ── save artifacts ────────────────────────────────────────────
def _trunc(s, n=600):
    return s if len(s) <= n else s[:n] + "…[truncated]"

_per_rows = []
for i, row in enumerate(eval_rows):
    _per_rows.append({
        "test_row":               i,
        "id":                     row.get("id", ""),
        "gold_files":             row["output"]["touched_files"],
        "allowed_files":          row["input"]["allowed_edit_files"],
        "valid_diff_ft":          metrics["FT"]["valid_diff"][i],
        "valid_diff_base":        metrics["Base"]["valid_diff"][i],
        "no_prose_ft":            metrics["FT"]["no_prose"][i],
        "no_prose_base":          metrics["Base"]["no_prose"][i],
        "file_recall_ft":         metrics["FT"]["file_recall"][i],
        "file_recall_base":       metrics["Base"]["file_recall"][i],
        "scope_ft":               metrics["FT"]["scope_compliance"][i],
        "scope_base":             metrics["Base"]["scope_compliance"][i],
        "hunk_overlap_ft":        metrics["FT"]["hunk_overlap"][i],
        "hunk_overlap_base":      metrics["Base"]["hunk_overlap"][i],
        "geval_mean_ft":          metrics["FT"]["geval"][i]["mean"],
        "geval_mean_base":        metrics["Base"]["geval"][i]["mean"],
        "geval_detail_ft":        metrics["FT"]["geval"][i],
        "geval_detail_base":      metrics["Base"]["geval"][i],
        "gen_sec_ft":             metrics["FT"]["gen_sec"][i],
        "gen_sec_base":           metrics["Base"]["gen_sec"][i],
        "pred_preview_ft":        _trunc(metrics["FT"]["texts"][i]),
        "pred_preview_base":      _trunc(metrics["Base"]["texts"][i]),
        "gold_preview":           _trunc(row["output"]["unified_diff"]),
    })

_meta = {
    "created_utc":       datetime.datetime.utcnow().replace(microsecond=0).isoformat() + "Z",
    "eval_dir":          str(EVAL_OUT_DIR),
    "n_eval_rows":       n_total,
    "max_eval_rows_cap": MAX_EVAL_ROWS,
    "geval_model":       _GEVAL_MODEL,
    "geval_rubrics":     ["instruction_adherence", "scope_compliance",
                          "diff_quality", "syntax_and_style", "hallucination_penalty"],
    "adapter_dir":       str(ADAPTER_DIR),
    "base_model":        BASE_MODEL,
}

_readout_lines = [
    "=== AutoBot Patcher Eval ===",
    f"eval_dir={EVAL_OUT_DIR}",
    f"rows evaluated: {n_total}  (cap MAX_EVAL_ROWS={MAX_EVAL_ROWS})",
    f"G-Eval model: {_GEVAL_MODEL}  |  rubrics: 5",
    "",
]
for sr in summary_results:
    for k, v in sr.items():
        _readout_lines.append(f"  {k}: {v}")
    _readout_lines.append("")
_readout = "\n".join(_readout_lines)

_detailed = {
    "meta":            _meta,
    "metrics_readout": _readout,
    "summary_table":   summary_results,
    "per_test_row":    _per_rows,
    "plotting_hint": (
        "No PNGs written. Use per_test_row columns "
        "(valid_diff_ft/base, file_recall_ft/base, geval_mean_ft/base, "
        "geval_detail_ft.syntax_and_style, geval_detail_ft.hallucination_penalty) "
        "to chart Base vs FT deltas per rubric."
    ),
}

with open(EVAL_OUT_DIR / "patcher_eval_detailed_report.json",    "w") as f:
    json.dump(_detailed,        f, indent=2, default=str)
with open(EVAL_OUT_DIR / "patcher_eval_metrics_comparison.json", "w") as f:
    json.dump(summary_results,  f, indent=2, default=str)
with open(EVAL_OUT_DIR / "patcher_eval_run_meta.json",           "w") as f:
    json.dump(_meta,            f, indent=2, default=str)
with open(EVAL_OUT_DIR / "patcher_metrics_readout.txt",          "w") as f:
    f.write(_readout + "\n")

pd.DataFrame([{
    "test_row":     r["test_row"],
    "gen_sec_ft":   r["gen_sec_ft"],
    "gen_sec_base": r["gen_sec_base"],
} for r in _per_rows]).round(4).to_csv(
    EVAL_OUT_DIR / "patcher_per_row_latency.csv", index=False
)

print(_readout)
print(f"\nAll artifacts saved to: {EVAL_OUT_DIR}")

Eval output directory (reset): /content/drive/MyDrive/autobot_patcher/evals/patcher/v1
Eval rows: 150  (full eval JSONL has 2360; cap MAX_EVAL_ROWS=150)


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1332: UserWarning: You have requested `ensure_weight_tying`, but no tied modules are added in either `modules_to_save` or `target_modules`
  warnings.warn(


Loaded FT adapter from /content/drive/MyDrive/autobot_patcher/patcher_lora_adapter_a100

Running Base vs FT on 150 eval rows...



  0%|          | 0/150 [00:00<?, ?it/s]

...


Running G-Eval (OpenAI, 5 rubrics) on predictions...


G-Eval FT:   0%|          | 0/150 [00:00<?, ?it/s]

G-Eval Base:   0%|          | 0/150 [00:00<?, ?it/s]


PATCHER EVAL — FINAL PERFORMANCE COMPARISON
                       Model Valid Diff % No Prose % File Recall Scope Compliance Hunk Overlap G-Eval mean (1-5)   instruction_adherence   scope_compliance   diff_quality   syntax_and_style   hallucination_penalty Gen mean (s)
       Base (Qwen 2.5 Coder)      100.00%      0.00%      0.9222           0.9967       0.8350            4.7320                  4.8133             4.8333         4.5467             4.5733                  4.8933        27.99
Fine-Tuned (AutoBot Patcher)      100.00%     99.33%      0.9344           1.0000       0.8682            4.8773                  4.9733             4.8667         4.8800             4.8067                  4.8600        42.47
=== AutoBot Patcher Eval ===
eval_dir=/content/drive/MyDrive/autobot_patcher/evals/patcher/v1
rows evaluated: 150  (cap MAX_EVAL_ROWS=150)
G-Eval model: gpt-4o-mini  |  rubrics: 5

  Model: Base (Qwen 2.5 Coder)
  Valid Diff %: 100.00%
  No Prose %: 0.00%
  File Recall: 0.9

/tmp/ipykernel_856/2013343526.py:450: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_utc":       datetime.datetime.utcnow().replace(microsecond=0).isoformat() + "Z",


## Optional — push adapter to Hugging Face Hub

Uncomment and set `REPO_NAME` if you want to upload the LoRA adapter.

```python
# from huggingface_hub import login
# from google.colab import userdata
# login(token=userdata.get("HF_TOKEN"))
# REPO_NAME = "your-org/your-patcher-lora"
# trainer.model.push_to_hub(REPO_NAME)
# tokenizer.push_to_hub(REPO_NAME)
```